# Masked ST-Transformer BIG V3-FIXED — PEMS-BAY Benchmark (Anti-Cheat v4)
### Dataset: PEMS-BAY (325 nodes, 80% sparsity, 5 seeds). Zero-as-missing convention.

**Anti-cheat audit v4** — every model receives *only the same 20% observed sensor readings*
as input. No model sees the held-out 80% or future time steps.

### All fixes applied

| # | Fix | Models affected |
|---|-----|-----------------|
| v2-1 | BiLSTM bidirectional→forward | BiLSTM |
| v2-2 | BRITS backward GRU removed | BRITS-lite |
| v2-3 | SAITS/ASTGCN causal attention mask | SAITS-lite, ASTGCN-lite |
| v3-1 | Unified numpy RNG for eval masks | ALL models |
| v3-2 | Eval chunk size = training window (48) | Our model |
| v4-1 | Node embedding added | MLP, SAITS-lite |
| v4-2 | KNN masked-distance (only observed dims) | KNN |
| v4-3 | Graph models: HA fill for unobserved nodes | DCRNN, GWN, ASTGCN |

**v4-1 rationale:** at 80% sparsity, unobserved nodes receive input `[0, 0, sin, cos]`.
Without a node embedding, MLP and SAITS cannot distinguish sensors and collapse to
a global prediction — explaining their worse-than-HA results.

**v4-3 rationale:** graph diffusion was spreading masked zeros into neighbouring nodes,
corrupting the spatial signal. Filling with the HA prior gives a neutral, informative
starting point while the mask feature still tells the model which nodes are real.


In [13]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU memory ready.")

GPU memory ready.


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import glob
import pickle
import urllib.request
import warnings

warnings.filterwarnings("ignore")

GLOBAL_SEED = 42
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_NAME = 'PEMS-BAY'
SPARSITY = 0.80
BATCH_TIME = 48
HIDDEN_DIM = 96
N_LAYERS = 5
N_HEADS = 4
DROPOUT = 0.1
TRAIN_EPOCHS = 1200
STEPS_PER_DAY = 288
EVAL_SEEDS = [42, 43, 44, 45, 46]
HUBER_BETA = 1.0

PEMSBAY_CSV_URL = "https://zenodo.org/records/5146275/files/PEMS-BAY.csv?download=1"
PEMSBAY_ADJ_URL = "https://zenodo.org/records/5146275/files/adj_mx_bay.pkl?download=1"

def find_file(candidates, search_roots=('.', '/kaggle/input', '/kaggle/working')):
    cand_lower = [c.lower() for c in candidates]
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for path in glob.glob(os.path.join(root, '**', '*'), recursive=True):
            if os.path.isfile(path) and os.path.basename(path).lower() in cand_lower:
                return path
    return None

def download_if_missing(url, dest):
    if not os.path.exists(dest):
        print(f"Downloading {dest} from {url} ...")
        urllib.request.urlretrieve(url, dest)
    return dest

def load_speed_array():
    """Return speed_raw [T, N] np.float32 from .h5 or .csv."""
    h5 = find_file(['pems-bay.h5', 'PEMS-BAY.h5'])
    if h5 is not None:
        print(f"  H5: {h5}")
        df = pd.read_hdf(h5)
        return np.nan_to_num(df.values.astype(np.float32), nan=0.0)
    csv = find_file(['pems-bay.csv', 'PEMS-BAY.csv'])
    if csv is None:
        csv = download_if_missing(PEMSBAY_CSV_URL, 'PEMS-BAY.csv')
    print(f"  CSV (timeseries): {csv}")
    df = pd.read_csv(csv, index_col=0)
    return np.nan_to_num(df.values.astype(np.float32), nan=0.0)

def load_adjacency(num_nodes):
    """Return adjacency matrix [N, N] from DCRNN-format .pkl. PEMS-BAY-specific filenames."""
    pkl = find_file(['adj_mx_bay.pkl', 'adj_mx_pems_bay.pkl'])
    if pkl is None:
        pkl = download_if_missing(PEMSBAY_ADJ_URL, 'adj_mx_bay.pkl')
    print(f"  Adj PKL: {pkl}")
    with open(pkl, 'rb') as f:
        obj = pickle.load(f, encoding='latin1')
    adj_mx = obj[2] if isinstance(obj, (list, tuple)) and len(obj) >= 3 else obj
    adj_mx = np.asarray(adj_mx, dtype=np.float32)
    if adj_mx.shape[0] != num_nodes:
        raise ValueError(
            f"Adjacency shape {adj_mx.shape} doesn't match data ({num_nodes} nodes). "
            f"Wrong pickle picked up -- delete cached adj_mx*.pkl and re-run."
        )
    adj = (adj_mx > 0.1).astype(np.float32)
    np.fill_diagonal(adj, 0)
    return adj

print(f"Device: {device} | Dataset: {DATASET_NAME} | Target Sparsity: {SPARSITY*100}%")

Device: cuda | Dataset: PEMS-BAY | Target Sparsity: 80.0%


In [15]:
class STBlock(nn.Module):
    def __init__(self, hidden, n_heads, ff_mult=2, dropout=0.0):
        super().__init__()
        self.temp = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * ff_mult, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.spat = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * ff_mult, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)

    def forward(self, h, spatial_pad_mask=None):
        B, N, T, H = h.shape
        cm = torch.triu(torch.full((T, T), float('-inf'), device=h.device), diagonal=1)
        h = self.temp(h.reshape(B * N, T, H), src_mask=cm).reshape(B, N, T, H)
        h = h.permute(0, 2, 1, 3).contiguous().reshape(B * T, N, H)
        if spatial_pad_mask is not None:
            h = self.spat(h, src_key_padding_mask=spatial_pad_mask)
        else:
            h = self.spat(h)
        h = h.reshape(B, T, N, H).permute(0, 2, 1, 3).contiguous()
        return h


class MaskedSTTransformerV3(nn.Module):
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=64, n_heads=4, n_layers=3, max_T=288, dropout=0.0,
                 nmean_trust_thresh=0.05):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.nmean_trust_thresh = nmean_trust_thresh
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)  # [N]
        self.register_buffer('node_stds', node_stds_t)    # [N]

        # 6 input features: x, m, t_sin, t_cos, n_mean, locf
        self.in_proj = nn.Linear(6, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb = nn.Parameter(torch.randn(max_T, hidden) * 0.02)

        self.blocks = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)

        self.meta_gate = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Linear(hidden, 1)
        self.last_alpha = None
        self.last_nmean_trust_frac = None  # diagnostic: fraction of positions where n_mean is trusted

    def _compute_locf(self, x, m):
        B, N, T = x.shape
        locf = torch.zeros_like(x)
        current_val = torch.zeros(B, N, device=x.device)  # default to mean (z-score = 0)
        for t in range(T):
            obs_t = x[:, :, t]
            mask_t = m[:, :, t]
            current_val = torch.where(mask_t > 0.5, obs_t, current_val)
            locf[:, :, t] = current_val
        return locf

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1)
        std_v = self.node_stds.view(1, -1, 1)

        # 1. Compute causal LOCF on the fly
        locf = self._compute_locf(x, m)
        locf_kmh = locf * std_v + mean_v

        # 2. Compute spatial neighbor mean over robust LOCF speeds
        adj_x_kmh = torch.matmul(self.adj_static, locf_kmh)
        adj_m = self.adj_static.sum(dim=-1, keepdim=True).view(1, -1, 1)
        n_mean_kmh = adj_x_kmh / (adj_m + 1e-6)
        n_mean = (n_mean_kmh - mean_v) / std_v
        
        self.last_nmean_trust_frac = torch.tensor(1.0, device=x.device) # always trusted

        # Stack 6 input features: x, m, t_sin, t_cos, n_mean, locf
        feat = torch.stack([x, m, t_sin, t_cos, n_mean, locf], dim=-1)
        h = self.in_proj(feat)
        h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)

        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)

        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)

        residual = self.residual_head(h).squeeze(-1)
        
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs = adj_m_obs / (self.adj_static.sum(dim=-1, keepdim=True) + 1e-8)
        alpha = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()
        
        # 3. Add learned residual correction directly to LOCF-imputed speed!
        return locf + alpha * residual

In [16]:
print("Loading data...")
speed_raw = load_speed_array()[:5000]
NUM_NODES = speed_raw.shape[1]
TRAIN_END = 4000
print(f"  Detected: T={speed_raw.shape[0]}, N={NUM_NODES}")

# PEMS-BAY: 0 = missing reading. Build a validity mask and apply it everywhere.
valid_raw = (speed_raw > 0).astype(np.float32)
print(f"  Valid fraction: {valid_raw.mean():.3f}")

# Per-node mean/std over valid entries only
node_means = np.zeros(NUM_NODES, dtype=np.float32)
node_stds = np.ones(NUM_NODES, dtype=np.float32)
for n in range(NUM_NODES):
    vals = speed_raw[:TRAIN_END, n][valid_raw[:TRAIN_END, n] > 0]
    if len(vals) > 0:
        node_means[n] = vals.mean()
        node_stds[n] = vals.std() + 1e-8

speed_norm = (speed_raw - node_means) / node_stds

# HA prior per (node, time-of-day), averaged over valid entries only
slot_idx = np.arange(len(speed_norm)) % STEPS_PER_DAY
tod_mean = np.zeros((NUM_NODES, STEPS_PER_DAY), dtype=np.float32)
for s in range(STEPS_PER_DAY):
    sel = slot_idx[:TRAIN_END] == s
    sub_data = speed_norm[:TRAIN_END][sel]
    sub_valid = valid_raw[:TRAIN_END][sel]
    sums = (sub_data * sub_valid).sum(axis=0)
    cnts = sub_valid.sum(axis=0) + 1e-8
    tod_mean[:, s] = sums / cnts

ha_prior = torch.tensor(tod_mean[:, slot_idx].T, dtype=torch.float32).to(device)
speed_gpu = torch.tensor(speed_norm, dtype=torch.float32).to(device)
valid_gpu = torch.tensor(valid_raw, dtype=torch.float32).to(device)
node_means_t = torch.tensor(node_means, dtype=torch.float32).to(device)
node_stds_t = torch.tensor(node_stds, dtype=torch.float32).to(device)

adj = load_adjacency(NUM_NODES)
D = np.diag(1.0 / np.sqrt(adj.sum(axis=1) + 1e-8))
adj_norm = D @ adj @ D
A_t = torch.tensor(adj_norm, dtype=torch.float32).to(device)

print(f"Data and adjacency ready. NUM_NODES={NUM_NODES}, avg degree={adj.sum(1).mean():.2f}, edges={int(adj.sum())}")

Loading data...
  CSV (timeseries): ./PEMS-BAY.csv
  Detected: T=5000, N=325
  Valid fraction: 1.000
  Adj PKL: ./adj_mx_bay.pkl
Data and adjacency ready. NUM_NODES=325, avg degree=7.29, edges=2369


## Baseline Models for Comparison (Fair, Causal Evaluation)

All models receive **only the 20% observed sensor readings** (`x * m_eff`) as input.
No model has access to the held-out 80% or to future time steps.
Models marked **[fixed]** had future-leakage bugs that have now been corrected.

| # | Model | Family | Graph? | Causal? |
|---|-------|--------|--------|---------|
| 1 | Historical Average (HA) | Statistical | No | ✓ |
| 2 | LOCF | Statistical | No | ✓ |
| 3 | Global Mean | Statistical | No | ✓ |
| 4 | Ridge Regression | Linear | No | ✓ |
| 5 | KNN Imputer (spatial, train-set neighbors only) | Non-param | No | ✓ |
| 6 | MLP | Neural | No | ✓ |
| 7 | LSTM | RNN | No | ✓ |
| 8 | BiLSTM → 2L-LSTM **[fixed]** | RNN | No | ✓ |
| 9 | GRU | RNN | No | ✓ |
| 10 | TCN (causal dilated) | Conv | No | ✓ |
| 11 | SAITS-lite **[fixed]** | Transformer | No | ✓ |
| 12 | BRITS-lite **[fixed]** | RNN+decay | No | ✓ |
| 13 | DCRNN-lite | GNN+RNN | Yes | ✓ |
| 14 | ASTGCN-lite **[fixed]** | GNN+Attn | Yes | ✓ |
| 15 | GWN-lite | GNN+TCN | Yes | ✓ |
| 16 | **MaskedSTTransformerV3-FIXED (ours)** | ST-Attn | Yes | ✓ |


In [17]:
import numpy as np, torch

RESULTS = {}

# ── Shared mask generator (ALL models must use this) ──────────────────────
# Uses numpy default_rng so masks are identical regardless of torch state.
def make_eval_mask_np(seed, EL, N):
    """Return bool mask [EL, N] — True = observed (20%), False = held-out (80%)."""
    rng = np.random.default_rng(seed)
    return (rng.random((EL, N)) > SPARSITY).astype(np.float32)  # [T, N]

# ── Shared eval helper (numpy, for non-graph models) ──────────────────────
def eval_fn(pred_fn, label):
    ES, EL = 4500, 450
    x_ev = speed_norm[ES:ES+EL]   # [T,N]
    v_ev = valid_raw [ES:ES+EL]
    maes = []
    for seed in EVAL_SEEDS:
        m_ev = make_eval_mask_np(seed, EL, NUM_NODES)  # [T,N]
        sm   = (m_ev == 0) & (v_ev > 0)
        if not sm.any(): continue
        idx  = np.arange(ES, ES+EL)
        p    = np.clip(pred_fn(x_ev.T, v_ev.T, m_ev.T, idx), 0, 120)
        t    = np.clip(x_ev.T * node_stds[:,None] + node_means[:,None], 0, 120)
        maes.append(np.abs(p[sm.T] - t[sm.T]).mean())
    arr = np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label] = arr

# 1. Historical Average
eval_fn(lambda xn,vn,mn,idx: tod_mean[:,idx%STEPS_PER_DAY]*node_stds[:,None]+node_means[:,None],
        "1. Historical Average (HA)")

# 2. LOCF
def locf(xn, vn, mn, idx):
    N, T = xn.shape
    out  = tod_mean[:,idx%STEPS_PER_DAY]*node_stds[:,None]+node_means[:,None]
    last = out[:,0].copy()
    for t in range(T):
        obs  = (mn[:,t]>0)&(vn[:,t]>0)
        last = np.where(obs, xn[:,t]*node_stds+node_means, last)
        out[:,t] = last
    return out
eval_fn(locf, "2. LOCF (Last-Obs Carried Forward)")

# 3. Global mean
eval_fn(lambda xn,vn,mn,idx: np.broadcast_to(node_means[:,None],xn.shape).copy(),
        "3. Global Mean (per-node train mean)")


1. Historical Average (HA)                            MAE: 3.1198 +/- 0.0051
2. LOCF (Last-Obs Carried Forward)                    MAE: 1.9592 +/- 0.0123
3. Global Mean (per-node train mean)                  MAE: 5.5168 +/- 0.0072


In [18]:
from sklearn.linear_model import Ridge
import warnings; warnings.filterwarnings("ignore")

# 4. Ridge Regression (unchanged)
print("Fitting Ridge regressors...")
ridge_models = []
for n in range(NUM_NODES):
    si  = slot_idx[:TRAIN_END]
    Xtr = np.column_stack([np.sin(2*np.pi*si/STEPS_PER_DAY),
                           np.cos(2*np.pi*si/STEPS_PER_DAY),
                           tod_mean[n, si]])
    ytr = speed_norm[:TRAIN_END, n]
    mk  = valid_raw[:TRAIN_END, n] > 0
    clf = Ridge(alpha=1.0); clf.fit(Xtr[mk], ytr[mk])
    ridge_models.append(clf)

def ridge(xn, vn, mn, idx):
    N, T = xn.shape; out = np.zeros((N,T),dtype=np.float32)
    for n in range(N):
        Xn = np.column_stack([np.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY),
                              np.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY),
                              tod_mean[n, idx%STEPS_PER_DAY]])
        out[n] = ridge_models[n].predict(Xn)*node_stds[n]+node_means[n]
    return out
eval_fn(ridge, "4. Node-wise Ridge Regression")

# 5. KNN Imputer — causal spatial imputation with masked Euclidean distance.
#
# Why the previous fix failed: at 80% sparsity, a 325-dim query has ~260 zeros
# (unobserved sensors set to 0). Standard Euclidean distance treats those zeros
# as signal, so neighbours are found based on the zero pattern, not speed values.
#
# Fix: compute distance only over the ~65 observed dimensions, normalised by
# the number of shared observed sensors. This is the standard 'missing-value
# aware' nearest-neighbour approach used in the imputation literature.
# Causal: each timestep is treated independently (spatial, not temporal KNN).
print("Fitting KNN (masked-distance spatial imputer)...")
tr_norm = speed_norm[:TRAIN_END].copy()           # [T_tr, N] z-scored
tr_valid = (valid_raw[:TRAIN_END] > 0).astype(np.float32)  # [T_tr, N]
# Keep only training rows with >=10% valid sensors
enough    = tr_valid.mean(axis=1) >= 0.10
tr_ref    = tr_norm[enough]          # [M, N]
tv_ref    = tr_valid[enough]         # [M, N]  1=valid, 0=missing
K_NEIGH   = 5

def knn_masked(xn, vn, mn, idx):
    """Causal spatial KNN with masked Euclidean distance.
    xn,vn,mn: [N,T] numpy arrays (z-scored / valid / observed-mask).
    Returns [N,T] in km/h.
    Each timestep is solved independently — no future used.
    """
    N, T = xn.shape
    out = (tod_mean[:, idx % STEPS_PER_DAY] * node_stds[:, None]
           + node_means[:, None]).copy()   # HA fallback [N,T]
    for t in range(T):
        obs = (mn[:, t] > 0) & (vn[:, t] > 0)   # [N] bool — truly observed
        if obs.sum() < 2:
            continue   # too few observed sensors — keep HA
        q_vals = xn[:, t]          # [N] z-scored query row

        # Masked squared distance: average over shared valid dimensions only
        # shared[m,n] = 1 if both query and ref row m have sensor n valid
        shared = tv_ref * obs.astype(np.float32)  # [M, N]
        n_shared = shared.sum(axis=1) + 1e-8      # [M]
        diff = (tr_ref - q_vals) * shared         # [M, N] — zero out unshared
        dist = (diff ** 2).sum(axis=1) / n_shared  # [M] mean-sq dist over shared

        # k nearest neighbours
        knn_idx = np.argpartition(dist, K_NEIGH)[:K_NEIGH]  # [K]
        neigh_v  = tr_ref[knn_idx]    # [K, N] z-scored
        neigh_ok = tv_ref[knn_idx]    # [K, N] validity

        # Fill: observed sensors keep their value; missing → weighted avg of neighbours
        neigh_sum = (neigh_v * neigh_ok).sum(0)     # [N]
        neigh_cnt = neigh_ok.sum(0) + 1e-8          # [N]
        imputed   = neigh_sum / neigh_cnt            # [N] z-scored
        row_norm  = np.where(obs, q_vals, imputed)   # [N] z-scored
        out[:, t] = np.clip(row_norm * node_stds + node_means, 0, 120)
    return out

eval_fn(knn_masked, "5. KNN Imputer (k=5, masked-dist)")


Fitting Ridge regressors...
4. Node-wise Ridge Regression                         MAE: 3.1204 +/- 0.0051
Fitting KNN (masked-distance spatial imputer)...
5. KNN Imputer (k=5, masked-dist)                     MAE: 2.1841 +/- 0.0143


In [19]:
import torch.nn as nn
import torch.nn.functional as F

NE, NH, NLR, NBT = 1200, 64, 1e-3, 48

def train_nw(cls, label, **kw):
    net = cls(4, NH, **kw).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=NLR)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NE)
    xg, vg = speed_gpu[:TRAIN_END], valid_gpu[:TRAIN_END]
    T_tr, N = xg.shape
    for ep in range(1, NE+1):
        net.train()
        t0 = np.random.randint(0, T_tr-NBT)
        xb, vb = xg[t0:t0+NBT], vg[t0:t0+NBT]
        idx = torch.arange(t0, t0+NBT, device=device)
        s   = torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
        c   = torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
        mb  = (torch.rand(NBT, N, device=device) > SPARSITY).float()
        me  = mb * vb; xi = xb * me
        feat = torch.stack([xi.T, me.T,
            s.unsqueeze(0).expand(N, -1),
            c.unsqueeze(0).expand(N, -1)], dim=-1)   # [N,T,4]
        pred = net(feat).squeeze(-1)                  # [N,T]
        lm = (mb.T == 0) & (vb.T > 0)
        if not lm.any(): continue
        loss = F.smooth_l1_loss(pred[lm], xb.T[lm])  # z-scored targets
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        opt.step(); sch.step()
    print(f"  {label} trained."); return net

def eval_nw(net, label):
    """Uses make_eval_mask_np — identical masks across all models."""
    net.eval()
    ES, EL = 4500, 450
    xev = speed_gpu[ES:ES+EL]; vev = valid_gpu[ES:ES+EL]
    idx = torch.arange(ES, ES+EL, device=device)
    s = torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
    c = torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
    st  = torch.tensor(node_stds,  device=device)
    mn_ = torch.tensor(node_means, device=device)
    maes = []
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_ev = torch.tensor(
                make_eval_mask_np(seed, EL, NUM_NODES), device=device)  # [T,N]
            me   = m_ev * vev
            feat = torch.stack([
                (xev * me).T, me.T,
                s.unsqueeze(0).expand(NUM_NODES, -1),
                c.unsqueeze(0).expand(NUM_NODES, -1)], dim=-1)  # [N,T,4]
            pred = net(feat).squeeze(-1)               # [N,T]
            pk   = (pred * st[:, None] + mn_[:, None]).clamp(0, 120)
            tk   = (xev.T * st[:, None] + mn_[:, None]).clamp(0, 120)
            sm   = (m_ev.T == 0) & (vev.T > 0)
            maes.append(torch.abs(pk[sm] - tk[sm]).mean().item())
    arr = np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label] = arr

# ── MLP with node embedding ───────────────────────────────────────────────
# ROOT CAUSE FIX: at 80% sparsity, an unobserved node's input is [0,0,sin,cos]
# — indistinguishable from any other unobserved node at the same timestep.
# Without a node identity, the MLP predicts the same value for all missing
# nodes, behaving like a global mean. Adding a learned node embedding gives
# each sensor a unique fingerprint even when its value is masked out.
class MLP(nn.Module):
    def __init__(self, F, H, n_nodes=325, **k):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, H)
        self.proj = nn.Linear(F, H)
        self.net = nn.Sequential(
            nn.LayerNorm(H * 2),
            nn.Linear(H * 2, H * 2), nn.GELU(),
            nn.LayerNorm(H * 2),
            nn.Linear(H * 2, H),     nn.GELU(),
            nn.Linear(H, 1))
        self._n = n_nodes

    def forward(self, x):   # x: [N, T, F]
        N, T, _ = x.shape
        nids = torch.arange(N, device=x.device)
        ne   = self.node_emb(nids).unsqueeze(1).expand(N, T, -1)  # [N,T,H]
        hf   = self.proj(x)                                        # [N,T,H]
        return self.net(torch.cat([hf, ne], dim=-1))               # [N,T,1]

class LSTM(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.r=nn.LSTM(F,H,batch_first=True); self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

# FIXED (v2): bidirectional=True saw future; replaced with 2-layer forward LSTM.
class BiLSTM(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__()
        self.r=nn.LSTM(F, H, num_layers=2, batch_first=True, dropout=0.1)
        self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

class GRUNet(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.r=nn.GRU(F,H,batch_first=True); self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

eval_nw(train_nw(MLP,    "MLP",    n_nodes=NUM_NODES), "6.  MLP (per-node + node-emb)")
eval_nw(train_nw(LSTM,   "LSTM"),                      "7.  LSTM (per-node)")
eval_nw(train_nw(BiLSTM, "BiLSTM"),                    "8.  BiLSTM->2L-LSTM (causal, fixed)")
eval_nw(train_nw(GRUNet, "GRU"),                       "9.  GRU (per-node)")


  MLP trained.
6.  MLP (per-node + node-emb)                         MAE: 3.5799 +/- 0.0054
  LSTM trained.
7.  LSTM (per-node)                                   MAE: 2.2654 +/- 0.0194
  BiLSTM trained.
8.  BiLSTM->2L-LSTM (causal, fixed)                   MAE: 2.2226 +/- 0.0172
  GRU trained.
9.  GRU (per-node)                                    MAE: 2.3004 +/- 0.0274


In [20]:
# 10. TCN — causal dilated convolutions: no fix needed.
class CausalConv(nn.Module):
    def __init__(self,c,k,d):
        super().__init__(); self.p=(k-1)*d; self.c=nn.Conv1d(c,c,k,dilation=d)
    def forward(self,x): return self.c(F.pad(x,(self.p,0)))

class TCNet(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__()
        self.proj=nn.Linear(F,H)
        self.convs=nn.ModuleList([CausalConv(H,3,2**i) for i in range(4)])
        self.norms=nn.ModuleList([nn.LayerNorm(H) for _ in range(4)])
        self.head=nn.Linear(H,1)
    def forward(self,x):
        h=self.proj(x).permute(0,2,1)
        for cv,nm in zip(self.convs,self.norms):
            r=h; h=F.gelu(nm(cv(h).permute(0,2,1))).permute(0,2,1)+r
        return self.head(h.permute(0,2,1))
eval_nw(train_nw(TCNet,"TCN"), "10. TCN (causal dilated, per-node)")

# 11. SAITS-lite — FIXED v2 (causal mask) + FIXED v4 (node embedding).
# ROOT CAUSE FIX: same as MLP — at 80% sparsity, unobserved nodes all have
# input [0,0,sin,cos]. Without node identity the transformer cannot distinguish
# sensors and collapses to a global prediction. Node embedding added.
class SAITSLite(nn.Module):
    def __init__(self, F, H, n_nodes=325, **k):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, H)
        self.proj = nn.Linear(F, H)
        kw = dict(nhead=4, dim_feedforward=H*2, dropout=0.1,
                  activation='gelu', batch_first=True, norm_first=True)
        self.a1 = nn.TransformerEncoderLayer(H, **kw)
        self.a2 = nn.TransformerEncoderLayer(H, **kw)
        self.h1 = nn.Linear(H, 1); self.h2 = nn.Linear(H, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self._n = n_nodes

    @staticmethod
    def _cmask(T, device):
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self, x):   # x: [N, T, F]
        N, T, _ = x.shape
        cm  = self._cmask(T, x.device)
        ne  = self.node_emb(torch.arange(N, device=x.device))  # [N, H]
        h   = self.proj(x) + ne.unsqueeze(1)                   # [N, T, H]
        h1  = self.a1(h,  src_mask=cm)
        h2  = self.a2(h1, src_mask=cm)
        a   = torch.sigmoid(self.alpha)
        return a * self.h1(h1) + (1-a) * self.h2(h2)

eval_nw(train_nw(SAITSLite, "SAITS", n_nodes=NUM_NODES),
        "11. SAITS-lite (causal Attn + node-emb, fixed)")

# 12. BRITS-lite — forward GRU only (fixed v2). No node-emb needed:
# GRU hidden state h carries node-specific temporal history, so the model
# accumulates a unique fingerprint per node through recurrence.
class BRITSLite(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.H=H
        self.gf   = nn.GRU(F*2, H, batch_first=True)
        self.impf  = nn.Linear(H, F)
        self.head  = nn.Linear(H, 1)

    def _run(self,x,m,gru,imp):
        N,T,Ff=x.shape
        h=torch.zeros(1,N,self.H,device=x.device); outs=[]
        for t in range(T):
            xh  = imp(h.squeeze(0))
            xc  = m[:,t,:]*x[:,t,:] + (1-m[:,t,:])*xh
            o,h = gru(torch.cat([xc, m[:,t,:]], -1).unsqueeze(1), h)
            outs.append(o)
        return torch.cat(outs, 1)

    def forward(self,x):
        m  = x[:,:,1:2].expand_as(x)
        hf = self._run(x, m, self.gf, self.impf)
        return self.head(hf)
eval_nw(train_nw(BRITSLite,"BRITS"), "12. BRITS-lite (forward GRU only, fixed)")


  TCN trained.
10. TCN (causal dilated, per-node)                    MAE: 1.9895 +/- 0.0200
  SAITS trained.
11. SAITS-lite (causal Attn + node-emb, fixed)        MAE: 3.4992 +/- 0.0290
  BRITS trained.
12. BRITS-lite (forward GRU only, fixed)              MAE: 1.9662 +/- 0.0220


In [21]:
# Graph baselines: feat [B,N,T,F] + adj A -> [B,N,T]
GE, GH, GLR = 1200, 64, 1e-3

def train_g(net, label):
    # ROOT CAUSE FIX for graph models: unobserved nodes previously had x=0
    # fed into graph diffusion, spreading zeros into neighbouring nodes'
    # representations. Fix: fill unobserved node values with HA prior
    # (z-scored). The mask feature still tells the model which nodes are real.
    opt=torch.optim.Adam(net.parameters(),lr=GLR)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=GE)
    xg,vg=speed_gpu[:TRAIN_END],valid_gpu[:TRAIN_END]; T_tr,N=xg.shape
    for ep in range(1,GE+1):
        net.train()
        t0=np.random.randint(0,T_tr-48)
        xb=xg[t0:t0+48].T.unsqueeze(0); vb=vg[t0:t0+48].T.unsqueeze(0)
        ha=ha_prior[t0:t0+48].T.unsqueeze(0)   # [1,N,48] z-scored HA prior
        idx=torch.arange(t0,t0+48,device=device)
        s=torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,N,-1)
        c=torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,N,-1)
        mb=(torch.rand(1,N,48,device=device)>SPARSITY).float(); me=mb*vb
        # ▶ FIX: unobserved positions filled with HA (not 0) before graph diffusion
        xi = xb*me + ha*(1-me)   # observed: real value; unobserved: HA prior
        feat=torch.stack([xi[0],me[0],s[0],c[0]],dim=-1).unsqueeze(0)
        pred=net(feat,A_t)
        lm=(mb==0)&(vb>0)
        if not lm.any(): continue
        loss=F.smooth_l1_loss(pred[lm],xb[lm])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),0.5)
        opt.step(); sch.step()
    print(f"  {label} trained."); return net

def eval_g(net, label):
    net.eval()
    ES,EL=4500,450
    xev=speed_gpu[ES:ES+EL].T.unsqueeze(0); vev=valid_gpu[ES:ES+EL].T.unsqueeze(0)
    hae=ha_prior[ES:ES+EL].T.unsqueeze(0)   # [1,N,EL] HA prior for fill
    idx=torch.arange(ES,ES+EL,device=device)
    s=torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    c=torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st=torch.tensor(node_stds,device=device).view(1,-1,1)
    mn_=torch.tensor(node_means,device=device).view(1,-1,1)
    maes=[]
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_np = make_eval_mask_np(seed, EL, NUM_NODES)  # [T,N]
            mev=torch.tensor(m_np, device=device).T.unsqueeze(0)  # [1,N,T]
            me=mev*vev
            # ▶ FIX: fill unobserved with HA prior (not 0) before graph diffusion
            xi = xev*me + hae*(1-me)
            feat=torch.stack([xi[0],me[0],s[0],c[0]],dim=-1).unsqueeze(0)
            pred=net(feat,A_t)
            pk=(pred*st+mn_).clamp(0,120); tk=(xev*st+mn_).clamp(0,120)
            sm=(mev==0)&(vev>0)
            maes.append(torch.abs(pk[sm]-tk[sm]).mean().item())
    arr=np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label]=arr

class DiffGCN(nn.Module):
    def __init__(self,i,o,K=2):
        super().__init__(); self.K=K; self.l=nn.Linear((K+1)*i,o)
    def forward(self,x,A):
        out=[x]; Ax=x
        for _ in range(self.K): Ax=torch.matmul(A.unsqueeze(0),Ax); out.append(Ax)
        return self.l(torch.cat(out,-1))

# 13. DCRNN-lite — forward GRU + graph diffusion: strictly causal, no fix needed.
class DCRNNLite(nn.Module):
    def __init__(self,F=4,H=64,**k):
        super().__init__(); self.H=H
        self.gr=DiffGCN(F+H,H); self.gu=DiffGCN(F+H,H); self.gc=DiffGCN(F+H,H)
        self.head=nn.Linear(H,1)
    def _step(self,x,h,A):
        xu=torch.cat([x,h],-1)
        r=torch.sigmoid(self.gr(xu,A)); u=torch.sigmoid(self.gu(xu,A))
        c=torch.tanh(self.gc(torch.cat([x,r*h],-1),A))
        return u*h+(1-u)*c
    def forward(self,feat,A):
        B,N,T,_=feat.shape; h=torch.zeros(B,N,self.H,device=feat.device); outs=[]
        for t in range(T): h=self._step(feat[:,:,t,:],h,A); outs.append(self.head(h))
        return torch.stack(outs,2).squeeze(-1)
dcrnn=DCRNNLite(H=GH).to(device); train_g(dcrnn,"DCRNN-lite"); eval_g(dcrnn,"13. DCRNN-lite (DiffGCN + GRU)")

# 14. ASTGCN-lite — FIXED: added causal mask to temporal self-attention.
# Original used full attention with no mask → position t attended to t+k (future leakage).
# Fix: same upper-triangular -inf causal mask as SAITS fix above.
class ASTGCNLite(nn.Module):
    def __init__(self,F=4,H=64,**k):
        super().__init__()
        self.proj=nn.Linear(F,H); self.gcn=DiffGCN(H,H)
        self.ta=nn.TransformerEncoderLayer(H,4,H*2,dropout=0.1,
            activation="gelu",batch_first=True,norm_first=True)
        self.head=nn.Linear(H,1)

    @staticmethod
    def _cmask(T, device):
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self,feat,A):
        B,N,T,_=feat.shape; h=self.proj(feat)
        hs=h.permute(0,2,1,3).reshape(B*T,N,-1)
        hs=self.gcn(hs,A).reshape(B,T,N,-1).permute(0,2,1,3)
        h=h+hs; cm=self._cmask(T, feat.device)
        ht=self.ta(h.reshape(B*N,T,-1), src_mask=cm).reshape(B,N,T,-1)  # causal
        return self.head(h+ht).squeeze(-1)
astgcn=ASTGCNLite(H=GH).to(device); train_g(astgcn,"ASTGCN-lite"); eval_g(astgcn,"14. ASTGCN-lite (GCN + Causal Attn, fixed)")

# 15. GWN-lite — causal dilated convolutions: strictly left-to-right, no fix needed.
class GWNLite(nn.Module):
    def __init__(self,F=4,H=64,num_nodes=325,**k):
        super().__init__()
        self.E1=nn.Parameter(torch.randn(num_nodes,10))
        self.E2=nn.Parameter(torch.randn(10,num_nodes))
        self.proj=nn.Linear(F,H)
        self.convs=nn.ModuleList([nn.Conv1d(H,H*2,2,dilation=2**i) for i in range(4)])
        self.gcn=DiffGCN(H,H,K=1); self.head=nn.Linear(H,1)
    def forward(self,feat,A_static):
        B,N,T,_=feat.shape
        Aa=torch.softmax(torch.relu(self.E1@self.E2),-1)
        Am=0.5*(A_static+Aa)
        h=self.proj(feat).permute(0,1,3,2).reshape(B*N,-1,T)
        skip=[]
        for cv in self.convs:
            d=cv.dilation[0]; p=(cv.kernel_size[0]-1)*d
            g=cv(F.pad(h,(p,0))); g1,g2=g.chunk(2,1)
            s=torch.tanh(g1)*torch.sigmoid(g2)
            h=h+s[:,:,:T]; skip.append(h)
        h=sum(skip).reshape(B,N,-1,T).permute(0,3,1,2).reshape(B*T,N,-1)
        h=self.gcn(h,Am).reshape(B,T,N,-1).permute(0,2,1,3)
        return self.head(h).squeeze(-1)
gwn=GWNLite(H=GH,num_nodes=NUM_NODES).to(device); train_g(gwn,"GWN-lite"); eval_g(gwn,"15. GWN-lite (Adaptive GCN + Gated TCN)")


  DCRNN-lite trained.
13. DCRNN-lite (DiffGCN + GRU)                        MAE: 3.1279 +/- 0.0169
  ASTGCN-lite trained.
14. ASTGCN-lite (GCN + Causal Attn, fixed)            MAE: 3.6502 +/- 0.0215
  GWN-lite trained.
15. GWN-lite (Adaptive GCN + Gated TCN)               MAE: 2.7709 +/- 0.0066


In [22]:
net = MaskedSTTransformerV3(NUM_NODES, A_t, node_means_t, node_stds_t,
                            hidden=HIDDEN_DIM, n_heads=N_HEADS,
                            n_layers=N_LAYERS, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)

n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
BATCH_SIZE = 2
print(f"Training ST-Transformer BIG V3-FIXED [{DATASET_NAME}] | params: {n_params/1e6:.2f}M | "
      f"hidden={HIDDEN_DIM} layers={N_LAYERS} dropout={DROPOUT} epochs={TRAIN_EPOCHS} | "
      f"loss=Huber(beta={HUBER_BETA}) | features=5 (guarded km/h n_mean) | mask-aware spatial attn")
for ep in range(1, TRAIN_EPOCHS + 1):
    net.train()

    t0_list = np.random.randint(0, TRAIN_END - BATCH_TIME, BATCH_SIZE)
    x_list, ha_list, sin_list, cos_list, m_list, v_list = [], [], [], [], [], []

    for t0 in t0_list:
        x_list.append(speed_gpu[t0:t0+BATCH_TIME].T)
        ha_list.append(ha_prior[t0:t0+BATCH_TIME].T)
        v_list.append(valid_gpu[t0:t0+BATCH_TIME].T)
        t_idx = torch.arange(t0, t0 + BATCH_TIME, device=device)
        sin_list.append(torch.sin(2 * np.pi * (t_idx % STEPS_PER_DAY) / STEPS_PER_DAY).view(1, -1).expand(NUM_NODES, -1))
        cos_list.append(torch.cos(2 * np.pi * (t_idx % STEPS_PER_DAY) / STEPS_PER_DAY).view(1, -1).expand(NUM_NODES, -1))
        m_list.append((torch.rand(NUM_NODES, BATCH_TIME, device=device) > SPARSITY).float())

    x_batch = torch.stack(x_list)
    ha_batch = torch.stack(ha_list)
    sin_batch = torch.stack(sin_list)
    cos_batch = torch.stack(cos_list)
    m_batch = torch.stack(m_list)
    v_batch = torch.stack(v_list)

    # Effective mask: invalid positions are also "unobserved" so the model never
    # ingests a fake 0-km/h reading as a real observation.
    m_eff = m_batch * v_batch

    p = net(x_batch * m_eff, m_eff, sin_batch, cos_batch, ha_batch)

    # Score loss only on (random mask hides) AND (ground truth is valid)
    loss_mask = (m_batch == 0) & (v_batch > 0)
    if not loss_mask.any():
        continue
    loss = F.smooth_l1_loss(p[loss_mask], x_batch[loss_mask], beta=HUBER_BETA)
    optimizer.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
    optimizer.step(); scheduler.step()

    if ep % 100 == 0:
        a = net.last_alpha
        trust_frac = net.last_nmean_trust_frac.item() * 100
        sat0 = (a < 0.1).float().mean().item() * 100
        sat1 = (a > 0.9).float().mean().item() * 100
        print(f"Epoch {ep:4d} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.1e} | "
              f"alpha[mean={a.mean().item():.3f} max={a.max().item():.3f}] "
              f"sat<0.1: {sat0:4.1f}%  sat>0.9: {sat1:4.1f}%  nmean_trust: {trust_frac:4.1f}%")

Training ST-Transformer BIG V3-FIXED [PEMS-BAY] | params: 0.81M | hidden=96 layers=5 dropout=0.1 epochs=1200 | loss=Huber(beta=1.0) | features=5 (guarded km/h n_mean) | mask-aware spatial attn
Epoch  100 | Loss: 0.0313 | LR: 9.8e-04 | alpha[mean=0.256 max=0.374] sat<0.1:  0.0%  sat>0.9:  0.0%  nmean_trust: 100.0%
Epoch  200 | Loss: 0.1191 | LR: 9.3e-04 | alpha[mean=0.171 max=0.617] sat<0.1:  0.0%  sat>0.9:  0.0%  nmean_trust: 100.0%
Epoch  300 | Loss: 0.0241 | LR: 8.5e-04 | alpha[mean=0.193 max=0.410] sat<0.1:  0.0%  sat>0.9:  0.0%  nmean_trust: 100.0%
Epoch  400 | Loss: 0.0379 | LR: 7.5e-04 | alpha[mean=0.141 max=0.651] sat<0.1: 21.3%  sat>0.9:  0.0%  nmean_trust: 100.0%
Epoch  500 | Loss: 0.0736 | LR: 6.3e-04 | alpha[mean=0.173 max=0.842] sat<0.1: 15.4%  sat>0.9:  0.0%  nmean_trust: 100.0%
Epoch  600 | Loss: 0.0342 | LR: 5.0e-04 | alpha[mean=0.145 max=0.574] sat<0.1: 12.4%  sat>0.9:  0.0%  nmean_trust: 100.0%
Epoch  700 | Loss: 0.1002 | LR: 3.7e-04 | alpha[mean=0.155 max=0.635] sat<0

In [23]:
net.eval()
EVAL_START, EVAL_LEN = 4500, 450
# ▶ FIX: chunk size matches training window — LOCF context is consistent with training.
# Original CHUNK = BATCH_TIME*4 = 192 gave the LOCF anchor 4× more history than
# the model was trained with, artificially boosting its quality.
CHUNK = BATCH_TIME  # = 48, same as training

with torch.no_grad():
    x_eval = speed_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    ha_eval = ha_prior[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    v_eval = valid_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    t_idx = torch.arange(EVAL_START, EVAL_START + EVAL_LEN, device=device)
    t_sin = torch.sin(2 * np.pi * (t_idx % STEPS_PER_DAY) / STEPS_PER_DAY).view(1, 1, -1).expand(1, NUM_NODES, -1)
    t_cos = torch.cos(2 * np.pi * (t_idx % STEPS_PER_DAY) / STEPS_PER_DAY).view(1, 1, -1).expand(1, NUM_NODES, -1)

    stds  = torch.tensor(node_stds,  device=device).view(1, -1, 1)
    means = torch.tensor(node_means, device=device).view(1, -1, 1)

    mae_mod_seeds, mae_ha_seeds, scored_frac_seeds = [], [], []

    for seed in EVAL_SEEDS:
        # ▶ FIX: identical mask to all other models via shared numpy RNG
        m_np = make_eval_mask_np(seed, EVAL_LEN, NUM_NODES)  # [T,N]
        m_eval = torch.tensor(m_np, device=device).T.unsqueeze(0)  # [1,N,T]
        m_eff  = m_eval * v_eval

        preds = []
        for c0 in range(0, EVAL_LEN, CHUNK):
            c1 = min(c0 + CHUNK, EVAL_LEN)
            p_c = net(x_eval[:, :, c0:c1] * m_eff[:, :, c0:c1],
                      m_eff[:, :, c0:c1],
                      t_sin[:, :, c0:c1], t_cos[:, :, c0:c1],
                      ha_eval[:, :, c0:c1])
            preds.append(p_c)
        p_eval = torch.cat(preds, dim=2)

        p_kmh  = (p_eval * stds + means).clamp(0, 120)
        t_kmh  = (x_eval * stds + means).clamp(0, 120)
        ha_kmh = (ha_eval * stds + means).clamp(0, 120)

        # Score only on (random mask hides) AND (ground truth is real)
        score_mask = (m_eval == 0) & (v_eval > 0)
        n_scored   = int(score_mask.sum().item())
        n_held     = int((m_eval == 0).sum().item())
        scored_frac = n_scored / max(n_held, 1)

        mae_mod = torch.abs(p_kmh[score_mask] - t_kmh[score_mask]).mean().item()
        mae_ha  = torch.abs(ha_kmh[score_mask] - t_kmh[score_mask]).mean().item()
        mae_mod_seeds.append(mae_mod)
        mae_ha_seeds.append(mae_ha)
        scored_frac_seeds.append(scored_frac)
        print(f"seed {seed}: HA={mae_ha:.4f}  TFv3fix={mae_mod:.4f}  delta={mae_ha-mae_mod:+.4f}  "
              f"scored={n_scored}/{n_held} ({100*scored_frac:.1f}% valid)")

mae_mod_seeds = np.array(mae_mod_seeds)
mae_ha_seeds  = np.array(mae_ha_seeds)

print(f"\nFINAL RESULTS (80% Sparsity, ST-Transformer BIG V3-FIXED [{DATASET_NAME}], {len(EVAL_SEEDS)} seeds)")
print(f"Historical Average MAE: {mae_ha_seeds.mean():.4f} +/- {mae_ha_seeds.std():.4f} km/h")
print(f"V3-FIXED MAE:           {mae_mod_seeds.mean():.4f} +/- {mae_mod_seeds.std():.4f} km/h")
delta = mae_ha_seeds.mean() - mae_mod_seeds.mean()
print(f"Delta vs HA:            {delta:+.4f} km/h ({100*delta/mae_ha_seeds.mean():+.1f}%)")
print(f"Mean scored fraction:   {100*np.mean(scored_frac_seeds):.1f}%")


seed 42: HA=3.1243  TFv3fix=1.5628  delta=+1.5616  scored=117099/117099 (100.0% valid)
seed 43: HA=3.1122  TFv3fix=1.5641  delta=+1.5481  scored=117107/117107 (100.0% valid)
seed 44: HA=3.1239  TFv3fix=1.5913  delta=+1.5326  scored=116961/116961 (100.0% valid)
seed 45: HA=3.1152  TFv3fix=1.5841  delta=+1.5311  scored=116909/116909 (100.0% valid)
seed 46: HA=3.1235  TFv3fix=1.5850  delta=+1.5385  scored=117081/117081 (100.0% valid)

FINAL RESULTS (80% Sparsity, ST-Transformer BIG V3-FIXED [PEMS-BAY], 5 seeds)
Historical Average MAE: 3.1198 +/- 0.0051 km/h
V3-FIXED MAE:           1.5774 +/- 0.0117 km/h
Delta vs HA:            +1.5424 km/h (+49.4%)
Mean scored fraction:   100.0%


In [24]:
# Final comparison table
RESULTS["16. MaskedSTTransformerV3-FIXED (ours)"] = mae_mod_seeds

ha_mae = RESULTS.get("1. Historical Average (HA)", mae_ha_seeds).mean()
print()
print("="*72)
print(f"{'PEMS-BAY IMPUTATION BENCHMARK  (80% Sparsity, 5 seeds)':^72}")
print("="*72)
print(f"{'Model':<52} {'MAE':>8}  {'Std':>6}  {'vs HA':>7}")
print("-"*72)
for name, arr in sorted(RESULTS.items(), key=lambda kv: kv[1].mean()):
    tag = " < OURS" if "ours" in name else ""
    diff = arr.mean() - ha_mae
    print(f"{name:<52} {arr.mean():>8.4f}  {arr.std():>6.4f}  {diff:>+7.4f}{tag}")
print("-"*72)
ours  = RESULTS["16. MaskedSTTransformerV3-FIXED (ours)"].mean()
delta = ha_mae - ours
print(f"Ours vs HA: {delta:+.4f} km/h  ({100*delta/ha_mae:+.1f}%)")
print("="*72)



         PEMS-BAY IMPUTATION BENCHMARK  (80% Sparsity, 5 seeds)         
Model                                                     MAE     Std    vs HA
------------------------------------------------------------------------
16. MaskedSTTransformerV3-FIXED (ours)                 1.5774  0.0117  -1.5424 < OURS
2. LOCF (Last-Obs Carried Forward)                     1.9592  0.0123  -1.1606
12. BRITS-lite (forward GRU only, fixed)               1.9662  0.0220  -1.1537
10. TCN (causal dilated, per-node)                     1.9895  0.0200  -1.1303
5. KNN Imputer (k=5, masked-dist)                      2.1841  0.0143  -0.9357
8.  BiLSTM->2L-LSTM (causal, fixed)                    2.2226  0.0172  -0.8972
7.  LSTM (per-node)                                    2.2654  0.0194  -0.8544
9.  GRU (per-node)                                     2.3004  0.0274  -0.8195
15. GWN-lite (Adaptive GCN + Gated TCN)                2.7709  0.0066  -0.3489
1. Historical Average (HA)                             3